### 문제 3-1 :  콘텐츠분쟁해결 RAG 시스템 - 간단 실습 가이드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── 0단계: 문서 로드 ──────────────────────────
loader = PyPDFLoader('../data/콘텐츠분쟁해결_사례.pdf')

docs = loader.load()

print(f"로드된 페이지 수: {len(docs)}")
print(docs[0].metadata)          # 어떤 정보가 붙어있는지 확인

# ── 1단계: 문서 분할 ──────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        #  법률 사례는 1200-1800자 권장
    chunk_overlap=300,      #  사례 맥락 보존을 위해 200-400자
    separators=[
        "\n【사건개요】",     #  법률 문서 섹션 구분자
        "\n【쟁점사항】",     #  쟁점 부분 구분
        "\n【처리경위】",     #  처리 과정 구분
        "\n【처리결과】",     #  결과 부분 구분
        "\n■", "\n\n", "\n", ".", " ", ""
    ]
)

chunks = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(chunks)}")
print(chunks[0].page_content[:300])   # 첫 청크가 사례 섹션 단위로 잘렸는지 눈으로 확인

In [ ]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()   # .env의 UPSTAGE_API_KEY 불러오기

# ── 2단계: 임베딩 모델 설정 ──────────────────────
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# ── 벡터DB 생성 (3단계 retriever가 쓸 vectorstore) ──
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"저장된 벡터 수: {vectorstore.index.ntotal}")   # 청크 수와 같아야 정상

In [ ]:
# ── 3단계: 검색기 설정 ──────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# ── 검색 테스트 (체인 만들기 전, 검색기만 단독 확인) ──
test_query = "게임 아이템이 사라졌는데 복구를 거부합니다"
results = retriever.invoke(test_query)

print(f"검색된 문서 수: {len(results)}")          
for i, doc in enumerate(results, 1):
    print(f"\n[{i}] 페이지 {doc.metadata.get('page')}")
    print(doc.page_content[:150])                

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate

# ── 4단계: LLM 설정 ─────────────────────────────
llm = ChatUpstage(
    model="solar-pro3",                 
    base_url="https://api.upstage.ai/v1",
    temperature=0.2 
)

# ── 5단계: 법률 자문 프롬프트 ───────────────────
prompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요                    
2. 관련 법령이나 조항이 있다면 명시하세요               
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요   
4. 실무적 해결방안을 단계별로 제시하세요               
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요  

전문 법률 조언:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# ── 확인: 빈칸 이름이 제대로 인식됐는지 ──────────
print(prompt.input_variables)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── 검색 결과 → 문자열 변환기 ──────────────────
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# ── 6단계: LCEL 체인 ──────────────────────────
qa_chain = (
    {
        "context": retriever | format_docs,          # TODO: 검색 → 문자열 변환 (위 그림 윗줄)
        "question": RunnablePassthrough()              # TODO: 질문을 그대로 통과 (위 그림 아랫줄)
    }
    | prompt                              # TODO: 빈칸 채우기
    | llm                              # TODO: 답변 생성
    | StrOutputParser()
)

# ── 단일 질문 테스트 ──────────────────────────
answer = qa_chain.invoke("온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?")
print(answer)

In [ ]:
# ── 7단계: 테스트 질문 ─────────────────────────
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"
]

for i, question in enumerate(test_questions, 1):
    print("=" * 60)
    print(f"[질문 {i}] {question}")
    print("-" * 60)
    answer = qa_chain.invoke(question)          # TODO: 6단계 단일 테스트에서 체인을 실행한 방법 그대로
    print(answer[:500]) 

In [20]:
# ── 8단계: 분쟁 유형 분류 함수 (선택) ───────────
def classify_dispute_type(query):
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]
    
    query_lower = query.lower().replace(" ", "")
    
    
    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"

# TODO: 질문 5개를 하나씩 분류해서 출력
# 힌트: 7단계 for문 구조를 그대로 재사용하되, 체인 대신 위 함수를 호출
for i, question in enumerate(test_questions, 1):
    print(f"[질문 {i}] 유형: {classify_dispute_type(question)}")

[질문 1] 유형: 게임
[질문 2] 유형: 이러닝
[질문 3] 유형: 웹콘텐츠
[질문 4] 유형: 게임
[질문 5] 유형: 이러닝
